# PySpark Partition Analysis — File Reading & Input Partitions

These notes explain how Spark determines the **number of input partitions when reading files**, using Parquet data as the practical example.

## Core idea

For file-based reads, the important Spark settings and factors discussed in this practical are:

- `spark.default.parallelism` / available executor CPU cores
- `spark.sql.files.maxPartitionBytes`
- `spark.sql.files.openCostInBytes`
- Total input file size

The practical focuses specifically on **input partition calculation**, not shuffle partition calculation.


## 1. Inspecting a Parquet File

The first practical reads one Parquet file and compares Spark's input partitions with Parquet row groups.

**Important distinction:**

- **Spark partition** → how Spark splits the input for parallel processing.
- **Parquet row group** → a storage-level structure inside a Parquet file.

The notebook also uses the Parquet footer through PyArrow to obtain row-group metadata without scanning all rows.

In [ ]:
# Read one parquet file only

from pyspark.sql import SparkSession
import pyarrow.parquet as pq

spark = SparkSession.builder.config("spark.sql.adaptive.enabled", "false").appName("parquet-inspect").getOrCreate()

path = "s3://ai-de-aws-learning-test/spark-learning/rides/tripdata/fhvhv_tripdata_2024-03.parquet" # file size is 450 MB

# --- Spark side ---
df = spark.read.parquet(path)

num_partitions = df.rdd.getNumPartitions()   # how Spark split the file for reading
total_records  = df.count()                  # full row count (triggers a scan)

# --- Parquet footer side (PyArrow) ---
pf = pq.ParquetFile(path)
num_row_groups   = pf.metadata.num_row_groups
rows_from_footer = pf.metadata.num_rows       # total rows straight from metadata (no scan)

print(f"Spark partitions : {num_partitions}")
print(f"Row groups       : {num_row_groups}")
print(f"Total records    : {total_records}")
print(f"Rows (footer)     : {rows_from_footer}")

for i in range(pf.metadata.num_row_groups):
    rg = pf.metadata.row_group(i)
    print(f"row group {i}: {rg.num_rows} rows, {rg.total_byte_size/1024/1024:.1f} MB")

## 2. How Spark Decides Input Partition Size

The practical uses the following formula:

```text
maxSplitBytes = min(
    maxPartitionBytes,
    max(openCostInBytes, totalBytes / totalExecutorCores)
)
```

Then, approximately:

```text
number of input partitions = totalBytes / maxSplitBytes
```

### Default values used in the notes

| Setting | Default discussed |
|---|---:|
| `spark.sql.files.maxPartitionBytes` | 128 MB |
| `spark.sql.files.openCostInBytes` | 4 MB |

`maxPartitionBytes` is the upper limit used when creating file splits.

`openCostInBytes` represents Spark's estimated cost of opening a file. It is especially important when there are many small files.

## 3. Example — Large Dataset

Given in the practical:

- Total input data = **1,376.4 MB** (`1,443,260,006` bytes)
- Available CPU cores = **8**
- `maxPartitionBytes` = **128 MB**
- `openCostInBytes` = **4 MB**

### Step 1 — Data per CPU core

```text
1,443,260,006 / 8
≈ 180,407,500 bytes
≈ 172 MB
```

### Step 2 — Calculate `maxSplitBytes`

```text
maxSplitBytes = min(128 MB, max(4 MB, 172 MB))
              = min(128 MB, 172 MB)
              = 128 MB
```

### Step 3 — Estimate partitions

```text
partitions ≈ 1,443,260,006 / 134,217,728
           ≈ 11 partitions
```

**Takeaway:** the 128 MB `maxPartitionBytes` limit becomes the controlling factor in this example.

In [ ]:
# path = "s3://ai-de-aws-learning-test/spark-learning/rides/tripdata/*.parquet"


# total 1376.4 mb of data, totalBytes = 1443260006 bytes
# available cpu cores = 8

# totalBytes/CPU_Cores = 1443260006/8 = 180407500 bytes = 172 mb

# maxPartitionBytes property is 128mb by default
# openCostInBytes default value is 4mb

# total partitions will be decided by maxSplitBytes = min(maxPartitionBytes, max(openCostInBytes , totalBytes/CPU_Cores) )

# maxSplitBytes = min ( 128mb , max(4mb , 172mb) ) = min ( 128mb , 172mb ) = 128mb

# partitions = totalBytes/maxSplitBytes = 1443260006/134217728 = 11 partitions


# you have 30 files, each one of it 1 mb = total size is 30 mb

# 1 files of 1 mb, when spark read it ( 1 mb + 4 mb ) = 5mb

# 30 * 5mb = 150 mb

# 150 mb (2 partitions)


## 4. Why `openCostInBytes` Matters

Suppose there are **30 files**, each of size **1 MB**.

Actual data:

```text
30 × 1 MB = 30 MB
```

With `openCostInBytes = 4 MB`, Spark treats each file as having an effective cost of roughly:

```text
1 MB + 4 MB = 5 MB
```

Therefore:

```text
30 × 5 MB = 150 MB effective size
```

Since one partition can hold up to 128 MB under the example settings, Spark needs approximately **2 input partitions**.

### Why is this useful?

Opening a file has overhead even when the file itself is tiny. On object storage such as S3, ADLS, or GCS, Spark must perform operations such as contacting storage, obtaining metadata, and opening the file. `openCostInBytes` lets Spark account for this cost when grouping small files into input partitions.


## 5. Check the Actual Spark Configuration

Use the following code to inspect the settings in the running Spark session.

In [ ]:
print("defaultParallelism  :", spark.sparkContext.defaultParallelism)
print("maxPartitionBytes   :", spark.conf.get("spark.sql.files.maxPartitionBytes"))
print("openCostInBytes     :", spark.conf.get("spark.sql.files.openCostInBytes"))

# These factors completely decides the number of partitions
# Actual formula is 

# maxSplitBytes = min( maxPartitionBytes,                 # 128 MB default
#                      max( openCostInBytes,              # 4 MB
#                           totalBytes / total executor cores ) )

# Total partitions = totalBytes / maxSplitBytes

# 1.) maxPartitionBytes - Maximum size of each partition

# 2.) openCostInBytes is a Spark configuration that tells Spark:
# “Assume opening a file has the same cost as reading this many bytes of data.”

# Why does Spark need this?

# Opening a file (especially on S3, ADLS, GCS, etc.) is not free. 
# Even if a file is only 10 KB, Spark still needs to:

# * Contact the storage service
# * Fetch metadata
# * Open a connection
# * Start reading

# This is mainly helpful for small files, For example

# You have 30 files, each 1 MB. Actual data size will be 30 MB
# Effective size Spark uses: 30 × (1 MB + 4 MB) = 150 MB because here Spark considering opening file of 1 MB size
# is equal to the efforts of opening 4 MB file

# Since one partition can hold only 128 MB, Spark creates 2 input partitions instead of 1.
# Without the open cost, Spark would have put all 30 files into a single partition.

# At first glance, yes, putting all 30 MB into one partition looks better because there is less scheduling overhead.

# But Spark isn’t optimizing only for data size—it’s optimizing for total work, which includes opening files.
# So having 2 partitions in this case will provide balanced parallelism


## 6. Changing `maxPartitionBytes`

The practical also demonstrates that changing `spark.sql.files.maxPartitionBytes` changes how Spark splits the input file for reading.

For example, setting it to **64 MB** creates smaller input splits than the default 128 MB configuration.

```python
spark.conf.set("spark.sql.files.maxPartitionBytes", str(64 * 1024 * 1024))
```

After changing the setting, the file should be read again so the new file-splitting configuration is applied.

In [ ]:
# We can set partition size this way

spark.conf.set("spark.sql.files.maxPartitionBytes", str(64 * 1024 * 1024))  # 64 MB
df = spark.read.parquet(path)
print(df.rdd.getNumPartitions())   # ~8 now

## 7. Quick Revision

### Input partition calculation

```text
                     Total input data
                            │
                            ▼
                 Total bytes / CPU cores
                            │
                            ▼
          max(openCostInBytes, bytes/core)
                            │
                            ▼
 min(maxPartitionBytes, value calculated above)
                            │
                            ▼
                    maxSplitBytes
                            │
                            ▼
              Number of input partitions
```

### Remember

1. `maxPartitionBytes` controls the maximum split size used for file reads.
2. `openCostInBytes` is particularly relevant when there are many small files.
3. Available CPU cores influence the calculated split target through `totalBytes / totalExecutorCores`.
4. These settings concern **input/file partitions**.
5. Shuffle partitions are controlled separately by `spark.sql.shuffle.partitions` and are not the focus of this notebook.
